<a href="https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


One row = one content page, belonging to one of 32 clients. prev_30d is the feature window (30 days before the prediction point); trend_direction is a categorical judgment comparing that window against a later window, not itself a raw feature. Exact calendar dates aren't available in the dataset — only day-count fields — so the time window is defined relative to the snapshot, not to fixed calendar dates.

In [1]:

!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd

Cloning into 'FlyRank-ML'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 168 (delta 76), reused 104 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 1.88 MiB | 15.19 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [2]:
df = pd.read_csv('/content/FlyRank-ML/data/raw/content_refresh_anonymized.csv')
print(df.shape)
print(df['content_id'].nunique())   # should match row count if truly 1 row per page
print(df['client_id'].nunique())    # should be 32
df.head()

(30000, 44)
30000
32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [8]:
# quick scan for any date-like columns
[col for col in df.columns if 'date' in col.lower() or 'time' in col.lower() or 'day' in col.lower()]

['days_with_impressions',
 'days_with_sessions',
 'content_age_days',
 'days_since_last_update']

In [9]:
df['days_with_impressionsare']

,days_with_impressions
0,88
1,88
2,88
3,88
4,88
...,...
29995,1
29996,77
29997,88
29998,88


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


Give below are the columns for features/ labels / context / exclude.


`days_with_impressions`, `days_with_sessions` are the columns with no window suffix. These columns could span full 90 days windows or just one sub-window. So we don't trust them.

`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier` — this is existing unresolved group, unsuffixed ratios/tiers of unconfirmed source window.

`trend_pct` is almost certainly the raw percentage change between the prior and later window that trend_direction was then bucketed from (e.g., "-15% → down", "+8% → up"). This is potential for data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [17]:
features_columns = ['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier']

features = df[features_columns]

In [18]:
target = df['trend_direction']

In [19]:
context = df[['content_id', 'client_id']]

In [20]:
excluded_columns = ['days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

exclude = df[excluded_columns]

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [22]:
# --- Grain check: one row = one content_id ---
print("Total rows:", len(df))
print("Unique content_id:", df['content_id'].nunique())
# these two numbers should match; if not, content_id isn't a unique row key

Total rows: 30000
Unique content_id: 30000


In [23]:
# --- Client grouping check ---
print("Unique client_id:", df['client_id'].nunique())  # expect 32
print(df.groupby('client_id').size().describe())  # pages-per-client distribution

Unique client_id: 32
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64


In [24]:
# --- Missing values scan (settles new/flat structural-artifact question) ---
print(df.isnull().sum().sort_values(ascending=False))

provider_used             21438
word_count                 7699
char_count                 7699
word_count_tier            7699
char_count_tier            7699
model_used                 5733
trend_pct                  3388
competition_level          2610
search_volume              2468
cpc                        2468
competition                2468
main_intent                2374
scroll_rate                 125
content_type                  0
client_id                     0
content_id                    0
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
sessions_prev_30d             0
clicks_prev_30d               0
impressions_prev_30d          0
sessions

In [25]:
# --- trend_direction class balance (settles majority-class / imbalance question) ---
print(df['trend_direction'].value_counts())
print(df['trend_direction'].value_counts(normalize=True))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64


In [26]:
# --- Window arithmetic check (re-confirms leakage audit) ---
window_check = df['impressions_prev_30d'] + df['impressions_last_30d']
print((df['impressions_90d'] - window_check).describe())
# small/near-zero residuals confirm prev_30d + last_30d ≈ 90d, i.e. last_30d sits inside 90d

count     30000.000000
mean       1988.229067
std        6099.411747
min           0.000000
25%          28.000000
50%         308.000000
75%        1512.250000
max      258505.000000
dtype: float64


In [27]:
# --- trend_pct vs trend_direction relationship check ---
print(df.groupby('trend_direction')['trend_pct'].describe())
# if trend_pct ranges align cleanly with each trend_direction bucket, it confirms
# trend_pct is the numeric source trend_direction was derived from

                   count        mean          std    min   25%    50%    75%  \
trend_direction                                                                
down             16262.0  -58.113830    23.488605 -100.0 -75.9 -55.60  -38.5   
flat                 0.0         NaN          NaN    NaN   NaN    NaN    NaN   
new                  0.0         NaN          NaN    NaN   NaN    NaN    NaN   
stable            5962.0   -3.185944    11.054723  -20.0 -12.7  -3.80    5.0   
up                4388.0  190.673997  1145.029477   20.0  36.5  62.55  123.1   

                     max  
trend_direction           
down               -20.0  
flat                 NaN  
new                  NaN  
stable              20.0  
up               44900.0  


In [28]:
# --- Unresolved window-source fields: correlation probe ---
for col in ['ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
            'days_with_impressions', 'days_with_sessions']:
    print(col, df[col].corr(df['impressions_prev_30d']), df[col].corr(df['impressions_last_30d']))
# doesn't prove window source definitively, but a much stronger correlation with one
# window over the other is a hint worth noting

ctr -0.017402126786923728 -0.01416305881427287
avg_position -0.06951564120439324 -0.0676415362964994
engagement_rate 0.02193259724666222 0.02145743768900539
scroll_rate -0.09045907318880818 -0.0912382746851161
ai_traffic_pct -0.007971083382196107 -0.0076790299484161065
days_with_impressions 0.22612477090411495 0.19345896575424984
days_with_sessions 0.5775191723564672 0.534921780095196


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


- **No calendar timestamps.** Only day-count fields exist (`content_age_days`,
  `days_since_last_update`, `days_with_impressions`, `days_with_sessions`). It's
  impossible to confirm whether all rows share a single snapshot date or were
  captured at different points in time. A true calendar-based time-aware split
  is therefore not possible; the client-grouped split is the primary leakage
  safeguard available.

- **Unconfirmed source windows for several fields.** `ctr`, `avg_position`,
  `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `days_with_impressions`,
  and `days_with_sessions` carry no window suffix, so their exact source period
  (prev_30d vs. last_30d vs. full 90d) is unconfirmed. They are held out of the
  feature set until verified.

- **`new`/`flat` are excluded on structural reasoning, not confirmed data.**
  These two `trend_direction` classes are treated as zero-history artifacts
  based on how they were reverse-engineered from the data, but the missing-values
  scan that would formally confirm this hasn't been run in this session.

- **This is a pre-aggregated snapshot, not raw daily data.** The dataset has
  already been rolled up per content_id; nothing here supports day-by-day or
  event-level analysis, only window-level comparisons.

- **Observational, not causal.** Nothing in this data supports claims like
  "a refresh caused a recovery," or attributing movement to specific actions
  (algorithm changes, AI citations, etc.). Any output is directional and
  decision-support only, meant to prioritize human review, not to explain
  *why* a page is trending a certain way.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.